# Data Cleaning and Preparation

Many researchers choose to do ``ad hoc`` processing of data from one form to another using a general-purpose programming language, like Python, Perl, R, or Java, or Unix text-processing tools like sed or awk. 

Here we discuss tools for missing data, duplicate data, string manipulation, and some other analytical data transformations.


## Handling Missing Data

Missing data occurs commonly in many data analysis applications. One of the goals of pandas is to make working with missing data as painless as possible. For example, all of the descriptive statistics on pandas objects exlude missing data by default. 

The way that missing data is represented in pandas objects is somewhat imperfect, but it is sufficient for most real-world use. For data with ``float64`` dtype, pandas uses the floating-point value ``NaN`` to represent missing data. 


We call this a *sentinel value*: when present, it indicates a missing (or *null*) value:

In [62]:
import numpy as np 
import pandas as pd 

float_data = pd.Series([1.2, -3.5, np.nan,0])

float_data

0    1.2
1   -3.5
2    NaN
3    0.0
dtype: float64

The ``isna`` method gives us a Boolean Series with ``True`` where values are null:

In [63]:
float_data.isna()

0    False
1    False
2     True
3    False
dtype: bool

When cleaning up data for analysis, it is often important to do analysis on the missing data itself to identify data collection problems biases in the data caused by missing data. 

The built-in Python ``None`` value is also treated as NA:

In [64]:
string_data = pd.Series(["aardvark", np.nan, None, "avocado"])

string_data

0    aardvark
1         NaN
2         NaN
3     avocado
dtype: str

In [65]:
string_data.isna()

0    False
1     True
2     True
3    False
dtype: bool

In [66]:
float_data = pd.Series([1, 2, None], dtype = 'float64')

float_data

0    1.0
1    2.0
2    NaN
dtype: float64

In [67]:
float_data.isna()

0    False
1    False
2     True
dtype: bool

List of some functions related to missing data handling:

| Method | Description |
|---|---|
| `dropna` | Filter axis labels based on whether values for each label have missing data, with varying thresholds for how much missing data to tolerate. |
|``fillna``|Fill missing data with some value or using an interpolation method such as ``"ffill"`` or ``"bfill"``|
|``isna``|Return Boolen values indicating which values are missing/NA.|
|``notna``|Negation of ``isna``, returns ``True`` for non-NA values and ``False`` for NA values|


## Filtering Out Missing Data

There are a few ways to filter out missing data. While you always have the option to do it by hand using ``pandas.isna`` and Boolean indexing, ``dropna`` can be helpful. On a Series, it returns the Series with only the nonnull data and index values:

In [68]:
data = pd.Series([1, np.nan, 3.5, np.nan, 7])

data.dropna()

0    1.0
2    3.5
4    7.0
dtype: float64

This is the same thing as doing:

In [69]:
data[data.notna()]

0    1.0
2    3.5
4    7.0
dtype: float64

With DataFrame objects, there are different ways to remove missing data. You may want to drop rows or columns that are all NA, or only those rows or columns containing any NAs at all. ``dropna`` by default drops any row containing a missing value:

In [70]:
data = pd.DataFrame([[1., 6.5, 3., ], [1., np.nan, np.nan], [np.nan, np.nan, np.nan], [np.nan, 6.5, 3.]])

data

,0,1,2
0,1.0,6.5,3.0
1,1.0,NaN,NaN
2,NaN,NaN,NaN
3,NaN,6.5,3.0


In [71]:
data.dropna()

,0,1,2
0,1.0,6.5,3.0


Passing ``how="all"`` will drop only rows that are all NA:

In [72]:
data.dropna(how="all")

,0,1,2
0,1.0,6.5,3.0
1,1.0,NaN,NaN
3,NaN,6.5,3.0


To drop columns in the same way, pass  ``axis = "columns"``:

In [73]:
data[4] = np.nan

data

,0,1,2,4
0,1.0,6.5,3.0,NaN
1,1.0,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN
3,NaN,6.5,3.0,NaN


In [74]:
data.dropna(axis="columns", how="all")

,0,1,2
0,1.0,6.5,3.0
1,1.0,NaN,NaN
2,NaN,NaN,NaN
3,NaN,6.5,3.0


Suppose you want to keep only rows containing at most a certain number of missing observations. You can indicate this with the ``thresh`` argument:

In [75]:
df = pd.DataFrame(np.random.standard_normal((7, 3)))

df.iloc[:4, 1] = np.nan

df.iloc[:2, 2] = np.nan

df

,0,1,2
0,0.559578,NaN,NaN
1,-0.065159,NaN,NaN
2,0.276948,NaN,-0.359711
3,-0.573385,NaN,0.329561
4,-0.520274,0.608510,-0.082505
5,0.462081,-0.419870,-0.762903
6,0.247502,0.185503,0.121763


In [76]:
df.dropna()

,0,1,2
4,-0.520274,0.608510,-0.082505
5,0.462081,-0.419870,-0.762903
6,0.247502,0.185503,0.121763


In [77]:
df.dropna(thresh=2)

,0,1,2
2,0.276948,NaN,-0.359711
3,-0.573385,NaN,0.329561
4,-0.520274,0.608510,-0.082505
5,0.462081,-0.419870,-0.762903
6,0.247502,0.185503,0.121763


## Filling in Missing Data 

Rather than filtering out missing data (and potentially discarding other data along with it), you may want to fill in the "holes" in any number of ways. For most purposes, the ``fillna`` method is the workhorse function to use. Calling ``fillna`` with a constant replaces missing values with that value:

In [78]:
df.fillna(0)

,0,1,2
0,0.559578,0.000000,0.000000
1,-0.065159,0.000000,0.000000
2,0.276948,0.000000,-0.359711
3,-0.573385,0.000000,0.329561
4,-0.520274,0.608510,-0.082505
5,0.462081,-0.419870,-0.762903
6,0.247502,0.185503,0.121763


Calling ``fillna`` with a dictionary, you can use a different full value for each column:

In [79]:
df.fillna({1: 0.5, 2:0})

,0,1,2
0,0.559578,0.500000,0.000000
1,-0.065159,0.500000,0.000000
2,0.276948,0.500000,-0.359711
3,-0.573385,0.500000,0.329561
4,-0.520274,0.608510,-0.082505
5,0.462081,-0.419870,-0.762903
6,0.247502,0.185503,0.121763


The same interpolation methods available for reindexing can be used with ``fillna``:


In [80]:
df = pd.DataFrame(np.random.standard_normal((6, 3)))

df.iloc[2:, 1] = np.nan

df.iloc[4:, 2] = np.nan

df

,0,1,2
0,-1.547968,-0.261666,0.225833
1,-0.039802,-1.907965,-1.135941
2,1.730347,NaN,1.842328
3,1.625896,NaN,-0.445778
4,0.628118,NaN,NaN
5,-1.177775,NaN,NaN


In [81]:
df.ffill()

,0,1,2
0,-1.547968,-0.261666,0.225833
1,-0.039802,-1.907965,-1.135941
2,1.730347,-1.907965,1.842328
3,1.625896,-1.907965,-0.445778
4,0.628118,-1.907965,-0.445778
5,-1.177775,-1.907965,-0.445778


In [82]:
df.ffill(limit=2)

,0,1,2
0,-1.547968,-0.261666,0.225833
1,-0.039802,-1.907965,-1.135941
2,1.730347,-1.907965,1.842328
3,1.625896,-1.907965,-0.445778
4,0.628118,NaN,-0.445778
5,-1.177775,NaN,-0.445778


With ``fillna`` you can do lots of other things such as simple data imputation using the median and mean statistics:

In [83]:
data = pd.Series([1., np.nan, 3.5, np.nan, 7])

data.fillna(data.mean())

0    1.000000
1    3.833333
2    3.500000
3    3.833333
4    7.000000
dtype: float64

# Data Transformation

Filtering, cleaning, and other transformations are another class of important operations. 

## Removing Duplicates 

Duplicate rows may be found in a DataFrame for any number of reasons. Here is an example:

In [84]:
data = pd.DataFrame({"k1": ["one", "two"] * 3 + ["two"],
                    "k2": [1, 1, 2, 3, 3, 4, 4]})
data

,k1,k2
0,one,1
1,two,1
2,one,2
3,two,3
4,one,3
5,two,4
6,two,4


The DataFrame method ``duplicated`` returns a Boolean Series indicating whether each row is a duplicate (its column values are exactly equal  to those in an earlier row) or not:

In [85]:
data.duplicated()

0    False
1    False
2    False
3    False
4    False
5    False
6     True
dtype: bool

Relatedly, ``drop_duplcates`` returns a DataFrame with rows where the ``duplicated`` array is ``False`` filtered out:

In [86]:
data.drop_duplicates()

,k1,k2
0,one,1
1,two,1
2,one,2
3,two,3
4,one,3
5,two,4


Both methods by default consider all of the columns; alternatively, you can specify any subset of them to detect duplicates. Suppose we had an additional column of values and wanted to filter duplicates based on the "k1" column:

In [87]:
data["v1"] = range(7)

data

,k1,k2,v1
0,one,1,0
1,two,1,1
2,one,2,2
3,two,3,3
4,one,3,4
5,two,4,5
6,two,4,6


In [88]:
data.drop_duplicates(subset=["k1"])

,k1,k2,v1
0,one,1,0
1,two,1,1


``duplicated`` and ``drop_duplicates`` by default keep the first observed value combination. Passing ``keep="last"`` will return the last one:

In [89]:
data.drop_duplicates(["k1", "k2"], keep="last")

,k1,k2,v1
0,one,1,0
1,two,1,1
2,one,2,2
3,two,3,3
4,one,3,4
6,two,4,6


## Transforming Data Using a Function or Mapping

For many datasets, you may wish to perform some transformation based on the values in the array, Series, or column in a DataFrame. Consider the following hypothetical data collected about various kinds of meat:

In [90]:
data = pd.DataFrame({"food": ["bacon", "pulled pork", "bacon",
                            "pastrami", "corned beef", "bacon",
                            "pastrami", "honey ham", "nova lox"],
                    "ounces": [4, 3, 12, 6, 7.5, 8, 3, 5, 6]})
data

,food,ounces
0,bacon,4.0
1,pulled pork,3.0
2,bacon,12.0
3,pastrami,6.0
4,corned beef,7.5
5,bacon,8.0
6,pastrami,3.0
7,honey ham,5.0
8,nova lox,6.0


Suppose you wanted to add a column indicating the type of animal that each food came from. Let's write down a mapping of each distinct meat type to the kind of animal:

In [91]:
meat_to_animal = {
"bacon": "pig",
"pulled pork": "pig",
"pastrami": "cow",
"corned beef": "cow",
"honey ham": "pig",
"nova lox": "salmon"
}

The ``map`` method on a Series accepts a function or dictionary-like object containing a mapping to do the transformation of values:

In [92]:
data["animal"] = data["food"].map(meat_to_animal)

data 

,food,ounces,animal
0,bacon,4.0,pig
1,pulled pork,3.0,pig
2,bacon,12.0,pig
3,pastrami,6.0,cow
4,corned beef,7.5,cow
5,bacon,8.0,pig
6,pastrami,3.0,cow
7,honey ham,5.0,pig
8,nova lox,6.0,salmon


We could also have passed a function that does all the work:

In [93]:
def get_animal(x):
    return meat_to_animal[x]

data["food"].map(get_animal)

0       pig
1       pig
2       pig
3       cow
4       cow
5       pig
6       cow
7       pig
8    salmon
Name: food, dtype: str

Using ``map`` is a convinient way to perform element-wise transformations and other data cleaning-relation operations.

## Replacing Values

Filling in missing data with the ``fillna`` method is a special case of more general value replacement. 

``map`` can be used to modify a subset of values in an object, but ``replace`` provides a simpler and more flexible way to do so. Let's consider this series:

In [94]:
data = pd.Series([1., -999., -1000., 3.])

data 

0       1.0
1    -999.0
2   -1000.0
3       3.0
dtype: float64

The ``-999`` values might be sentinel values for missing data. To replace these with NA values that pandas understands, we can use ``replace``, producing a new Series:

In [95]:
data.replace(-999, np.nan)



0       1.0
1       NaN
2   -1000.0
3       3.0
dtype: float64

If you want to replace multiple values at once, you instead pass a list and then the substitue value:

In [96]:
data.replace([-999, -1000], np.nan)

0    1.0
1    NaN
2    NaN
3    3.0
dtype: float64

To use a different replacement for each value, pass a list of substites:

In [97]:
data.replace([-999, -1000], [np.nan, 0])

0    1.0
1    NaN
2    0.0
3    3.0
dtype: float64

The argument passed can be a dictionary:

In [98]:
data.replace({-999: np.nan, -1000:0})

0    1.0
1    NaN
2    0.0
3    3.0
dtype: float64

## Renaming Axis Indexes

Like values in a Series, axis labels can be similarly transformed by a function or mapping of some form to produce new, differently labeled objects. You can alse modify the axes in place without creating a new data structure. Here's a simple example:

In [99]:
data = pd.DataFrame(np.arange(12).reshape((3, 4)), 
                    index=["Ohio", "Colorado", "New York"],
                    columns=["one", "two", "three", "four"])

data 

,one,two,three,four
Ohio,0,1,2,3
Colorado,4,5,6,7
New York,8,9,10,11


Like a Series, the axis indexes have a ``map`` method:

In [100]:
def transform(x):
    return x[:4].upper()

data.index.map(transform)

Index(['OHIO', 'COLO', 'NEW '], dtype='str')

You can assign to the ``index`` attribute, modifying the DataFrame in place:

In [101]:
data.index = data.index.map(transform)

data 

,one,two,three,four
OHIO,0,1,2,3
COLO,4,5,6,7
NEW,8,9,10,11


If you want to create a transformed version of a dataset without modifying the original, a useful method is ``rename``:

In [102]:
data.rename(index=str.title, columns=str.upper)

,ONE,TWO,THREE,FOUR
Ohio,0,1,2,3
Colo,4,5,6,7
New,8,9,10,11


Notably, ``rename`` can be used in conjunction with a dictionary-like object, providing new values for a subset of the axis labels:

In [103]:
data.rename(index={"OHIO": "INDIANA"}, columns={"three": "peekaboo"})



,one,two,peekaboo,four
INDIANA,0,1,2,3
COLO,4,5,6,7
NEW,8,9,10,11


``rename`` saves you from the chore of copying the DataFrame manually and assigning new values to its ``index`` and ``columns`` attributes.


## Discretization and Binning 

Continous data is often discretized or otherwise separated into "bins" for analysis. Suppose you have data about a group of people in a study, and you want to group them into a discrete age buckets:

In [104]:
ages = [20, 22, 25, 27, 21, 23, 37, 31, 61, 45, 41, 32]

Let's divide these into bins of 18 to 25, 26 to 35, 36 to 60, and finally 61 and older. To do so, you have to use ``pandas.cut``:

In [105]:
bins = [18, 25, 35, 60, 100]

In [107]:
age_categories = pd.cut(ages, bins)

age_categories

[(18, 25], (18, 25], (18, 25], (25, 35], (18, 25], ..., (25, 35], (60, 100], (35, 60], (35, 60], (25, 35]]
Length: 12
Categories (4, interval[int64, right]): [(18, 25] < (25, 35] < (35, 60] < (60, 100]]

The object pandas returns is a special Categorical object. The output you see describes the bins computed by ``pandas.cut``. Each bin is identified by a special interval value type containing the lower and upper limit of each bin:

In [108]:
age_categories.codes

array([0, 0, 0, 1, 0, 0, 2, 1, 3, 2, 2, 1], dtype=int8)

In [109]:
age_categories.categories

IntervalIndex([(18, 25], (25, 35], (35, 60], (60, 100]], dtype='interval[int64, right]')

In [110]:
age_categories.categories[0]

Interval(18, 25, closed='right')